In [3]:
import os
import json
import pandas as pd
import traceback

In [4]:
from dotenv import load_dotenv

load_dotenv()

my_key = os.getenv("HUGGING_FACE_API_KEY")
# print(my_key)


In [26]:
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEndpoint
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

In [30]:
# model = HuggingFaceEndpoint(
#     repo_id="mistralai/Mistral-7B-Instruct-v0.2",
#     task="conversational",
#     huggingfacehub_api_token=my_key,
#     temperature=0.7,
#     max_new_tokens=1000,
#     provider="auto"
# )

model = ChatOllama(model="llama3:latest")
output_parser = StrOutputParser()

In [31]:
prompt = PromptTemplate.from_template(
    "Can you suggest me a gift for an {occation} for a {relationship} who is {age} years old?. Give me answer directly in one sentence."
)

In [32]:
formatted1 = (prompt.format(occation="birthday", relationship="best friend", age="21"))
print(formatted1)

Can you suggest me a gift for an birthday for a best friend who is 21 years old?. Give me answer directly in one sentence.


In [33]:
llm_chain = prompt | model | output_parser
response1 = llm_chain.invoke({"occation": "birthday", "relationship": "best friend", "age": "21"})
print(response1)

Consider gifting your 21-year-old best friend an experience-based present, such as tickets to a concert or a sporting event, a fun activity like rock climbing or wine tasting, or a unique class or workshop that aligns with their interests.


In [46]:
RESPONSE_JSON = [
  {
    "question": "",
    "options": {
      "A": "",
      "B": "",
      "C": "",
      "D": ""
    },
    "correct_answer": "",
  }
]

In [36]:
TEMPLATE = """
Text: {TEXT}

You are an expert teacher generating MCQs.

Generate {num_questions} multiple-choice questions on the subject "{subject}" based on the text provided above.

Difficulty level: {difficulty}

Requirements:
- Each question must have 4 options.
- Only one correct answer.
- Avoid repeating concepts and questions.
- All the questions conforming to the text provided and the specified difficulty level.

Return the result ONLY in the following JSON format, you can also use it as a template for your response:
{RESPONSE_JSON}

Generate {num_questions} questions.
"""

In [ ]:
quiz_generation_prompt = PromptTemplate(
    template=TEMPLATE,
    input_variables=["num_questions", "subject", "difficulty", "TEXT", "RESPONSE_JSON"]
)


In [38]:
quiz_chain  = quiz_generation_prompt | model | output_parser

In [39]:
TEMPLATE1 = """
Text: {TEXT}

You are an expert teacher and academic reviewer.

Your task is to evaluate a set of multiple-choice questions (MCQs) created from the text above.

MCQs to evaluate:
{QUIZ}

Evaluation Criteria:
1. Verify whether each question is relevant to the provided text.
2. Check if the correct answer is actually correct.
3. Ensure the distractor options (incorrect choices) are plausible.
4. Identify if any questions are ambiguous, misleading, or poorly framed.
5. Detect repeated or very similar questions.
6. Verify that the difficulty level matches: {difficulty}.
7. Verify the grammatical and spelling correctness of the questions and options.

For each question:
- If a question is incorrect or poorly framed, rewrite it to improve clarity and correctness.
- Ensure the corrected question still follows the MCQ format.

For each option:
- If any grammatical or spelling errors are found, correct them.

Do not change the difficulty level of the questions.

Return the evaluation, after the corrections are made, strictly in the following JSON format:

{RESPONSE_JSON}


Only return JSON. Do not include additional commentary.
"""

In [40]:
quiz_evaluation_prompt =PromptTemplate(
    template=TEMPLATE1,
    input_variables=["TEXT", "QUIZ", "difficulty", "RESPONSE_JSON"]
)

In [44]:
evaluation_chain = quiz_evaluation_prompt | model | output_parser

In [ ]:
def full_pipeline(inputs):
    try:
        quiz_response = quiz_chain.invoke(inputs)
        print("Generated Quiz:")
        print(quiz_response)

        evaluation_inputs = {
            "TEXT": inputs["TEXT"],
            "QUIZ": quiz_response,
            "difficulty": inputs["difficulty"],
            "RESPONSE_JSON": RESPONSE_JSON
        }
        evaluation_response = evaluation_chain.invoke(evaluation_inputs)
        print("Evaluation and Corrections:")
        print(evaluation_response)

    except Exception as e:
        print("An error occurred during the pipeline execution:")
        traceback.print_exc()

In [42]:
test_file_path = "data.txt"

In [43]:
TEXT=""
with open(test_file_path, 'r') as file:
    TEXT = file.read()

print(TEXT)

# Supervised Learning

<br />

Supervised learning's tasks are well-defined and can be applied to a multitude
of scenarios---like identifying spam or predicting precipitation.

## Foundational supervised learning concepts

Supervised machine learning is based on the following core concepts:

- Data
- Model
- Training
- Evaluating
- Inference

### Data

Data is the driving force of ML. Data comes in the form of words and numbers
stored in tables, or as the values of pixels and waveforms captured in images
and audio files. We store related data in datasets. For example, we might have a
dataset of the following:

- Images of cats
- Housing prices
- Weather information

Datasets are made up of individual
[examples](https://developers.google.com/machine-learning/glossary#example) that contain
[features](https://developers.google.com/machine-learning/glossary#feature) and a
[label](https://developers.google.com/machine-learning/glossary#label). You could think of an example as
analogous to a

In [48]:

inputs = {
    "TEXT": TEXT,
    "num_questions": 7,
    "subject": "Machine Learning (Supervised Learning)",
    "difficulty": "medium",
    "RESPONSE_JSON": json.dumps(RESPONSE_JSON)
}

In [49]:
mcq_results = full_pipeline(inputs)

Generated Quiz:
Here are the 7 MCQs:

[{"question": "What is the driving force of ML?", 
"options": {"A": "Model", "B": "Data", "C": "Training", "D": "Inference"}, 
"correct_answer": "B"}],

[{"question": "What are labeled examples in supervised learning?", 
"options": {"A": "Examples without labels", "B": "Examples with features and labels", 
"C": "Examples only for training", "D": "Examples only for testing"}, 
"correct_answer": "B"}],

[{"question": "What is the goal of a model during training?", 
"options": {"A": "To predict the label from the features", 
"B": "To find the best solution to predict the labels", 
"C": "To update its solution based on the difference between predicted and actual values", 
"D": "To learn the mathematical relationship between the features and the label"}, 
"correct_answer": "D"}],

[{"question": "Why do large and diverse datasets produce a better model?", 
"options": {"A": "Because they provide more examples for training", 
"B": "Because they contain onl